# 04. market: 시장 객체를 이름으로 묶기, 시나리오, 시간

`src/quanttool/market/market.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `config`, `curves` (`build_curve`, `CurveFactors`, `evaluation_date`) | `build_market` → `Market`. 상품·시나리오·리스크가 모두 이것에 붙는다 |

**객체 지도** (이 순서로 본다)

1. `MarketObject` 규격
2. `build_market` 한 줄씩
3. `Market` 속성
4. 조회: `market[이름]`, `index`, `factors`
5. 시나리오: `apply`/`reset`/`scenario`
6. 평가일: `active`
7. 시간: `roll`

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.config import load_config
from quanttool.market.quotes import read_quotes

cfg = load_config(ROOT / "configs")
quotes = read_quotes(FIX / "market_sample.txt")
q = quotes[ASOF]

from quanttool.market import Market, MarketObject, build_market
from quanttool.market.curves import CurveFactors, build_curve

## `MarketObject` 규격
시장이 객체에 부르는 것은 이것뿐이다. 새 시장 객체(vol 서페이스 등)는 이것만 맞추면 시나리오·리스크가 그대로 동작한다.

In [ ]:
need = ["handle", "factors", "refresh", "roll"]
print(
    "규격:",
    list(MarketObject.__annotations__),
    [n for n in vars(MarketObject) if not n.startswith("_")],
)
cf = CurveFactors(
    build_curve(cfg.curves["UST"], cfg.conventions["UST"], q, ASOF)[0],
    ASOF,
    cfg.curves["UST"].factor_tenors,
)
{a: hasattr(cf, a) for a in need}

## `build_market(asof, quotes[asof], cfg)` 한 줄씩
그릇 셋 → 커브 반복(의존 순서) → 지수 → `Market`. 손으로 같은 일을 하고 결과를 비교한다.

In [ ]:
objects, handles, used = {}, {}, {}
for name in cfg.build_order():
    c = cfg.curves[name]
    curve, curve_quotes = build_curve(
        c, cfg.conventions[c.conventions], q, ASOF, False, handles, cfg.indices
    )
    used.update(curve_quotes)
    objects[name] = CurveFactors(curve, ASOF, c.factor_tenors)
    handles[name] = objects[name].handle
    print(f"{name:5} 시세 {len(curve_quotes):2}  이미 만든 Handle: {list(handles)}")
indices = {n: ic.make(n, handles[ic.curve]) for n, ic in cfg.indices.items()}
by_hand = Market(ASOF, objects, indices, used)

market = build_market(ASOF, q, cfg)
with market.active():
    print(
        "손으로 = build_market:",
        {n: by_hand[n].discount(5.0) - market[n].discount(5.0) for n in market.objects},
    )

## `Market` 속성

In [ ]:
pd.Series(
    {
        "asof": market.asof,
        "date (지금 평가일)": market.date,
        "objects": {n: type(o).__name__ for n, o in market.objects.items()},
        "quotes (시세 키 → Quote)": f"{len(market.quotes)}개, 예 {list(market.quotes)[:2]}",
        "indices": {n: type(i).__name__ for n, i in market.indices.items()},
        "current (적용 중 시나리오)": market.current,
        "_base (요인 기준값) 개수": len(market._base),
    }
)

`market.quotes`: 커브들이 쓴 시세 Quote를 시세 키로 모은 사전 (`read_quotes`의 하루치와 같은 키). `linked=True` 시장에서 값을 바꾸면 커브가 다시 풀린다 — 장중 시세 갱신 자리. 얼린 시장(기본)에서는 바꿔도 그대로다.

In [ ]:
live = build_market(ASOF, q, cfg, linked=True)
k = "IRS/RATE/KRW/CD91/5Y"
with live.active():
    before = live["CD91"].zeroRate(5.0, ql.Continuous).rate()
    live.quotes[k].setValue(live.quotes[k].value() + 0.001)
    print(
        f"linked: {k} +10bp → CD91 5Y 제로 {(live['CD91'].zeroRate(5.0, ql.Continuous).rate() - before) * 1e4:.2f}bp"
    )

## 조회
- `market[이름]` → 상품이 연결할 Handle (요인 층 커브)
- `market.index(이름)` → 예측 커브에 연결된 지수
- `market.factors()` → `{"커브/만기": Quote}` 시장 전체 요인

In [ ]:
f = market.factors()
print(
    type(market["UST"]).__name__,
    "|",
    market.index("CD91").name(),
    "|",
    len(f),
    "요인, 예",
    list(f)[:3],
)
try:
    market["KRW_GOVT"]
except KeyError as e:
    print(e)

## 시나리오: `apply` / `reset` / `with market.scenario(s)`
요인 = 기준 + 이동. 두 번 적용해도 **누적되지 않는다**.

In [ ]:
s = {"UST/5Y": 1e-4}
market.apply(s)
market.apply(s)
print("두 번 apply 뒤 UST/5Y:", f["UST/5Y"].value(), "| current:", market.current)
market.reset()
print("reset 뒤:", f["UST/5Y"].value(), market.current)
with market.scenario({"UST/5Y": 1e-4}):
    print("with 안:", f["UST/5Y"].value())
print("with 밖:", f["UST/5Y"].value())

지수는 예측 커브(CD91)를 따라간다. KOFR만 흔들면 CD91 지수 선도는 그대로다(커브는 얼려 있어 CD91을 다시 풀지 않는다).

In [ ]:
from quanttool.analysis.scenarios import parallel

idx = market.index("CD91")
fix = ql.SouthKorea(ql.SouthKorea.Settlement).adjust(ql.Date(23, 10, 2028))
rows = {}
for name, sc in {
    "base": {},
    "CD91 +10bp": parallel(market, "CD91", 1e-3),
    "KOFR +10bp": parallel(market, "KOFR", 1e-3),
}.items():
    with market.active(), market.scenario(sc):
        rows[name] = idx.fixing(fix)
pd.Series(rows, name="CD91 선도 (2028-10)")

## 평가일: `active()`
QuantLib 전역 평가일을 `market.date`로 잠시 둔다. 결제일·경과이자·지난 현금흐름·과거 fixing이 이 값으로 정해진다.

In [ ]:
S = ql.Settings.instance()
print("밖:", S.evaluationDate)
with market.active():
    print("안:", S.evaluationDate)

## 시간: `roll(d)`
객체마다 `roll(d)`(커브: ForwardForward), 그리고 [asof, d) 사이 지수 fixing을 오늘의 예측 선도로 채운다. asof로 돌아오면 지운다.

In [ ]:
d = date(2026, 10, 26)
first = ql.Date.from_date(ASOF)
with market.active():
    forward = idx.fixing(first, True)
market.roll(d)
with market.active():
    print(
        "date",
        market.date,
        "| UST 기준일",
        market["UST"].referenceDate(),
        "| 10/23 fixing 채움",
        idx.pastFixing(first) == forward,
    )
market.roll(ASOF)
print("되돌림:", market.date, idx.hasHistoricalFixing(first))

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `build_market(asof, quotes, cfg, linked)` | 날짜, 하루치 시세, 설정 | `Market` | 하루의 시장을 만들 때 |
| `market[이름]`, `index` | 이름 | Handle / 지수 | 상품 조립 (portfolio) |
| `factors()` | — | `{"객체/요인": Quote}` | 시나리오 키 확인 |
| `quotes` | — | `{시세 키: Quote}` | linked 시장의 시세 갱신 |
| `apply`/`reset`/`scenario` | `{요인: 이동}` | — | 흔들고 되돌리기 (risk) |
| `active()` | — | with 블록 | 상품 값을 볼 때 (npvs가 알아서) |
| `roll(d)` | 날짜 | — | theta, 시간 이동 |

## 내 메모

- 